In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import VectorAssembler

spark = SparkSession.builder.appName('MLlib').getOrCreate()

data = [(1, 5.0, 20.0), (2, 10.0, 25.0), (3, 15.0, 30.0), (4, 20.0, 35.0)]
columns = ['ID', 'Feature', 'Target']
df = spark.createDataFrame(data, columns)

assembler = VectorAssembler(inputCols=['Feature'], outputCol='Features')
df_transformed = assembler.transform(df)

lr = LinearRegression(featuresCol='Features', labelCol='Target')
model = lr.fit(df_transformed)


print(f'Coefficients: {model.coefficients}')
print(f'Intercept: {model.intercept}')

25/12/04 10:26:02 WARN Utils: Your hostname, yasa-IdeaPad-Slim-3-14IAH8 resolves to a loopback address: 127.0.1.1; using 192.168.1.6 instead (on interface wlp0s20f3)
25/12/04 10:26:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/04 10:26:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/04 10:26:05 WARN Instrumentation: [b61502cf] regParam is zero, which might cause numerical instability and overfitting.
25/12/04 10:26:07 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/12/04 10:26:07 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
25/12/04 10:26:07 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


Coefficients: [0.9999999999999992]
Intercept: 15.000000000000009


In [4]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.linalg import Vectors

data = [(1, 2.0, 3.0, 0), 
        (2, 1.0, 5.0, 1), 
        (3, 2.5, 4.5, 1), 
        (4, 3.0, 6.0, 0)]

columns = ['ID', 'f1', 'f2', 'Label']
df = spark.createDataFrame(data, columns)

assembler = VectorAssembler(
    inputCols=['f1', 'f2'],
    outputCol='Features'
)

df2 = assembler.transform(df)

lr = LogisticRegression(featuresCol='Features', labelCol='Label')
model = lr.fit(df2)

print("Coefficients:", model.coefficients)
print("Intercept:", model.intercept)

Coefficients: [-12.262057937838394,4.087352269372807]
Intercept: 11.568912735310269


In [6]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.linalg import Vectors, VectorUDT
from spyspark.sql.functions import udf
from pyspark.sql.types import *

data = [(1, [1.0, 1.0]), (2, [5.0, 5.0]), (3, [10.0, 10.0]), (4, [15.0, 15.0])]
columns = ['ID', 'Features']
df = spark.createDataFrame(data, columns)

to_vec = udf(lambda v: Vectors.dense(v), VectorUDT())
df2 = df.withColumn("Features", to_vec("Features"))

kmeans = KMeans(featuresCol='Features', k=2)
model = kmeans.fit(df2)

centers = model.clusterCenters()
print("Cluster centers:", centers)


Cluster centers: [array([12.5, 12.5]), array([3., 3.])]


In [2]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

spark = SparkSession.builder.appName("TitanicML").getOrCreate()

df = spark.read.csv("file:///home/yasa/Big Data/Titanic-Dataset.csv", header=True, inferSchema=True)

df.printSchema()
df.show(5)

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|   

In [4]:

df_clean = df.select("Survived", "Pclass", "Sex", "Age", "Fare", "SibSp", "Parch").dropna()


In [6]:
sex_indexer = StringIndexer(inputCol="Sex", outputCol="SexIndexed")

assembler = VectorAssembler(inputCols=["Pclass", "SexIndexed", "Age", "Fare", "SibSp", "Parch"],
                            outputCol="features")

pipeline = Pipeline(stages=[sex_indexer, assembler])
data = pipeline.fit(df_clean).transform(df_clean)

data.select("Survived", "features").show(5, truncate=False)

train, test = data.randomSplit([0.8, 0.2], seed=42)

lr = LogisticRegression(labelCol="Survived", featuresCol="features")
model = lr.fit(train)

predictions = model.transform(test)
predictions.select("Survived", "prediction", "probability").show(10)

evaluator = BinaryClassificationEvaluator(labelCol="Survived",rawPredictionCol="rawPrediction",
                                          metricName="areaUnderROC")

auc = evaluator.evaluate(predictions)
print("AUC =", auc)

+--------+------------------------------+
|Survived|features                      |
+--------+------------------------------+
|0       |[3.0,0.0,22.0,7.25,1.0,0.0]   |
|1       |[1.0,1.0,38.0,71.2833,1.0,0.0]|
|1       |[3.0,1.0,26.0,7.925,0.0,0.0]  |
|1       |[1.0,1.0,35.0,53.1,1.0,0.0]   |
|0       |[3.0,0.0,35.0,8.05,0.0,0.0]   |
+--------+------------------------------+
only showing top 5 rows



25/12/04 10:41:59 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/12/04 10:41:59 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


+--------+----------+--------------------+
|Survived|prediction|         probability|
+--------+----------+--------------------+
|       0|       1.0|[0.11846704248461...|
|       0|       1.0|[0.30496019894391...|
|       0|       1.0|[0.33094201062785...|
|       0|       1.0|[0.40021361974741...|
|       0|       1.0|[0.47115612100334...|
|       0|       0.0|[0.50946999024314...|
|       0|       0.0|[0.74128016753727...|
|       0|       0.0|[0.56388696341737...|
|       0|       0.0|[0.65731546987782...|
|       0|       0.0|[0.64965704044373...|
+--------+----------+--------------------+
only showing top 10 rows

AUC = 0.8750000000000004


In [9]:
paramGrid = (ParamGridBuilder().addGrid(lr.regParam, [0.01, 0.1, 0.5]).addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]).build())
cv = CrossValidator(estimator=lr, estimatorParamMaps=paramGrid, evaluator=evaluator, numFolds=5)

cv_model = cv.fit(train)

best_predictions = cv_model.transform(test)
best_auc = evaluator.evaluate(best_predictions)
print("Skor AUC", best_auc)

Skor AUC 0.8774691358024695
